# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pickle
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [3]:
#shape
spaceship.shape 

(8693, 14)

**Check for data types**

In [4]:
#types
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

**Check for missing values**

In [5]:
#null values
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [6]:
#drop rows containing any missing value (be careful, create a new df otherwise this does not work)
spaceship_clean = spaceship.dropna(axis="rows")

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [7]:
#replace cabin values
spaceship_clean['Cabin']=spaceship_clean['Cabin'].str[0]

C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_29284\3693897371.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spaceship_clean['Cabin']=spaceship_clean['Cabin'].str[0]


- Drop PassengerId and Name

In [8]:
#Drop PassengerId and Name
spaceship_clean = spaceship_clean.drop(columns = ["PassengerId","Name"])

In [9]:
# all columns
features = spaceship_clean.drop(columns=['Transported'], axis=1)
target = spaceship_clean["Transported"]

#here we make sure  that we gather all unique categorical values from our df columns in 'categories'. This way we avoid any missing and/or additional value
ohe = OneHotEncoder(sparse_output=False, drop='first', categories = [spaceship_clean[col].unique().tolist() for col in spaceship_clean.select_dtypes(include="object").columns])

#Perform Train Test Split
X_train, X_test, y_train, y_test = train_test_split(features,target, test_size = 0.20, random_state=0)

# X train dummies for categorical columns
X_train_cat = X_train.select_dtypes(include="object") 
X_test_cat = X_test.select_dtypes(include="object") 
ohe.fit(X_train_cat)  
X_train_cat_trans_np = ohe.transform(X_train_cat)
X_test_cat_trans_np = ohe.transform(X_test_cat)

# X train for numerical columns
X_train_num = X_train.select_dtypes(include="number") 
X_test_num = X_test.select_dtypes(include="number") 



In [10]:
# concatenation of X_cat & X_num before scaling
X_train_transformed = pd.concat([X_train_num.reset_index(drop=True), pd.DataFrame(X_train_cat_trans_np).reset_index(drop=True)], axis = 1)
X_test_transformed = pd.concat([X_test_num.reset_index(drop=True), pd.DataFrame(X_test_cat_trans_np).reset_index(drop=True)], axis = 1)
X_train_transformed.columns = X_train_transformed.columns.astype(str) 
X_test_transformed.columns = X_test_transformed.columns.astype(str) 

# apply scaling for all
# creation of the instance
std_scaler = StandardScaler()

# application to the values
std_scaler.fit(X_train_transformed)

with open("standard_scaler.pkl", "wb") as file:
    pickle.dump(std_scaler, file)

# transformation of the data
X_train_transformed_np = std_scaler.transform(X_train_transformed)
X_test_transformed_np  = std_scaler.transform(X_test_transformed)

# conversion into df
X_train_transformed_np_df = pd.DataFrame(X_train_transformed_np, columns=X_train_transformed.columns, index=X_train_transformed.index)
X_test_transformed_np_df  = pd.DataFrame(X_test_transformed_np, columns=X_test_transformed.columns, index=X_test_transformed.index)

knn_final = KNeighborsClassifier(n_neighbors=10)

knn_final.fit(X_train_transformed_np_df, y_train)

with open("knn_final.pkl", "wb") as file:
    pickle.dump(knn_final, file)

# apply scaling for all
# creation of the instance
std_scaler = StandardScaler()
# application to the values
std_scaler.fit(X_train_transformed)

with open("standard_scaler.pkl", "wb") as file:
    pickle.dump(std_scaler, file)

# transformation of the data
X_train_transformed_np = std_scaler.transform(X_train_transformed)
X_test_transformed_np  = std_scaler.transform(X_test_transformed)

# conversion into df
X_train_transformed_np_df = pd.DataFrame(X_train_transformed_np, columns=X_train_transformed.columns, index=X_train_transformed.index)
X_test_transformed_np_df  = pd.DataFrame(X_test_transformed_np, columns=X_test_transformed.columns, index=X_test_transformed.index)

knn_final = KNeighborsClassifier(n_neighbors=10)

knn_final.fit(X_train_transformed_np_df, y_train)

with open("knn_final.pkl", "wb") as file:
    pickle.dump(knn_final, file)

print(f"on train, model accuracy is: {knn_final.score(X_train_transformed_np_df, y_train)*100:.2f}%")
print(f"on test, model accuracy is: {knn_final.score(X_test_transformed_np_df, y_test)*100:.2f}%")


NameError: name 'KNeighborsClassifier' is not defined